# Test retrieval

The job `test_retrieval`, **run by hand** (not in `financebench_pipeline`, because the Vector Search endpoint is
billed by the hour while it exists). Everything is in this one notebook: there is not much to it.

1. Create the endpoint and a Delta Sync index on `gold_children`, from the stored vectors (`embedding`): nothing
   is embedded again.
2. Run the 150 FinanceBench questions, each in its own filing, through two LangChain retrievers:
   - `hybrid`: Vector Search's vector + keyword search
   - `ensemble`: `EnsembleRetriever` of a vector search and BM25 on the filing's parents

   Both are parent-child: the children are searched, the best child per parent is kept, and the parent's text
   (`parent_text`, synced into the index, not embedded) is what comes back. A fixed chunk is its own parent.
3. Score at k: the right filing, the evidence page, the evidence text (70% of its words), MRR. Saved in
   `test_retrieval_results`.
4. `finally`: delete the index, then the endpoint. `gold_children` and its embeddings are never touched.

FinanceBench's `evidence_page_num` is **0-based** (checked on the 3M 2018 10-K), gold's pages are 1-based, so 1 is
added. A cancelled run can leave the endpoint up: run again with `teardown_only = true`.

**Packages** are pinned and were checked together on Python 3.11. `mcp` is pinned because `mcp` 2.x breaks
`import databricks_langchain` (0.23.0).

In [ ]:
%pip install -q databricks-vectorsearch==0.82 databricks-langchain==0.23.0 langchain-classic==1.0.8 langchain-community==0.4.2 rank-bm25==0.2.2 mcp==1.26.0
dbutils.library.restartPython()

In [ ]:
import json
import re
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime, timezone

sys.path.insert(0, "../common")
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import DatabricksEmbeddings, DatabricksVectorSearch
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from pyspark.sql import functions as F

from names import Names

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("gold_schema", "finance_bench_gold")
dbutils.widgets.text("vector_search_endpoint", "financebench_vs")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en")
dbutils.widgets.text("embedding_dimension", "1024")
dbutils.widgets.text("strategies", "structured,fixed")
dbutils.widgets.text("modes", "hybrid,ensemble")
dbutils.widgets.text("ks", "1,3,5,10")
dbutils.widgets.dropdown("teardown", "true", ["true", "false"])
dbutils.widgets.dropdown("teardown_only", "false", ["true", "false"])

w = dbutils.widgets.get
n = Names(w("catalog"), w("bronze_schema"), w("silver_schema"), w("gold_schema"))
ENDPOINT, INDEX = w("vector_search_endpoint"), n.vector_index
strategies = [s.strip() for s in w("strategies").split(",") if s.strip()]
modes = [m.strip() for m in w("modes").split(",") if m.strip()]
ks = sorted(int(k) for k in w("ks").split(","))
K = ks[-1]
QUESTIONS_URL = "https://raw.githubusercontent.com/patronus-ai/financebench/main/data/financebench_open_source.jsonl"
# Synced into the index: filters (strategy, path), what a hit returns, and the vectors. Only `embedding` is searched.
RETURNED = ["chunk_id", "parent_id", "path", "heading_path", "page_start", "page_end", "parent_text"]
SYNCED = ["chunk_id", "strategy", "parent_id", "path", "company", "fiscal_period", "doc_type", "chunk_type",
          "heading_path", "page_start", "page_end", "embed_text", "parent_text", "embedding"]
vsc = VectorSearchClient(disable_notice=True)

## Endpoint and index

In [ ]:
def wait(check, what, timeout_s=3600):
    start = time.time()
    while True:
        done, state = check()
        print(f"  {what}: {state}")
        if done:
            return
        assert time.time() - start < timeout_s, f"{what} not ready after {timeout_s}s"
        time.sleep(20)


def create_index():
    """Endpoint and index, created if missing (the index's first sync starts with it), else synced. Waits."""
    if ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]:
        vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")

    def endpoint_online():
        state = vsc.get_endpoint(ENDPOINT).get("endpoint_status", {}).get("state")
        return state == "ONLINE", state
    wait(endpoint_online, f"endpoint {ENDPOINT}")

    try:
        index = vsc.get_index(endpoint_name=ENDPOINT, index_name=INDEX)
        index.sync()
    except Exception:  # not there yet
        index = vsc.create_delta_sync_index(
            endpoint_name=ENDPOINT, index_name=INDEX, source_table_name=n.children, pipeline_type="TRIGGERED",
            primary_key="chunk_id", embedding_vector_column="embedding",
            embedding_dimension=int(w("embedding_dimension")), columns_to_sync=SYNCED)

    def index_ready():
        status = index.describe().get("status", {})
        state = status.get("detailed_state", "UNKNOWN")
        assert "FAILED" not in state, f"index failed: {status.get('message')}"
        return bool(status.get("ready")) and "ONLINE" in state and "UPDATE" not in state, state
    wait(index_ready, f"index {INDEX}")


def teardown():
    """Delete the index, then the endpoint. The endpoint cannot go while its index is still being deleted, so that
    is retried. Raises if the endpoint is still there, so a run never ends green with an endpoint left running."""
    try:
        vsc.delete_index(endpoint_name=ENDPOINT, index_name=INDEX)
        print("index deleted")
    except Exception as e:
        print(f"index: {e}")
    for _ in range(10):
        try:
            vsc.delete_endpoint(name=ENDPOINT)
            print("endpoint deleted")
            return
        except Exception as e:
            if "not found" in str(e).lower() or "does not exist" in str(e).lower():
                print("endpoint already gone")
                return
            print(f"endpoint not deleted yet: {e}")
            time.sleep(30)
    raise RuntimeError(f"endpoint {ENDPOINT} is still there: delete it by hand (it is billed by the hour)")

## Retrievers

`parent_search` is the parent-child part: it searches 4 × k children (several children of one parent are often
next to each other) and keeps the best child per parent. Wrapped in a `RunnableLambda`, it is a LangChain
retriever, so `EnsembleRetriever` can merge it with BM25 (Reciprocal Rank Fusion, deduplicated by `parent_key`).
BM25 runs here, on the driver, so it is built on one filing's parents only.

In [ ]:
def as_parent(doc):
    """A child hit -> its parent: the parent's text, keyed by parent_id (a fixed chunk is its own parent)."""
    m = dict(doc.metadata)
    m["parent_key"] = m.get("parent_id") or m["chunk_id"]
    return Document(page_content=m.pop("parent_text", None) or doc.page_content, metadata=m)


def parent_search(store, path, strategy, query_type):
    def search(question):
        docs = store.similarity_search(question, k=K * 4, filter={"strategy": strategy, "path": path},
                                       query_type=query_type)
        out, seen = [], set()
        for d in map(as_parent, docs):
            if d.metadata["parent_key"] not in seen:
                seen.add(d.metadata["parent_key"])
                out.append(d)
        return out[:K]
    return RunnableLambda(search)


def retriever_for(store, mode, path, strategy):
    """The retriever for one filing and strategy."""
    if mode == "hybrid":
        return parent_search(store, path, strategy, "HYBRID")
    rows = (spark.table(n.children).filter((F.col("path") == path) & (F.col("strategy") == strategy))
            .select(F.coalesce("parent_id", "chunk_id").alias("parent_key"), "parent_text", "path",
                    "heading_path", "page_start", "page_end")
            .dropDuplicates(["parent_key"]).collect())
    parents = [Document(page_content=r.parent_text, metadata={k: v for k, v in r.asDict().items() if k != "parent_text"})
               for r in rows]
    return EnsembleRetriever(retrievers=[parent_search(store, path, strategy, "ANN"),
                                         BM25Retriever.from_documents(parents, k=K)],
                             weights=[0.5, 0.5], id_key="parent_key")

## Scores

Each score is the share of the questions, at each k. A question whose filing has no chunks counts as a miss.

| Score | A hit when |
| --- | --- |
| `doc_hit` | A chunk of the right filing is in the top k |
| `page_hit` (main) | A top-k chunk covers the evidence page (`page_start` ≤ page ≤ `page_end`) |
| `evidence_hit` | A top-k chunk has at least 70% of the words of the evidence text (the parser's text is not FinanceBench's) |
| `mrr` | 1 / rank of the first chunk that covers the evidence page, 0 if none in the top k |

In [ ]:
def words(text):
    return set(re.findall(r"[a-z0-9]+", (text or "").lower()))


def score(hits, evidence, k):
    def on_page(h, e):
        return h["doc"] == e["doc"] and h["page_start"] <= e["page"] <= h["page_end"]

    def has_text(h, e):
        want = words(e["text"])
        return h["doc"] == e["doc"] and bool(want) and len(want & words(h["text"])) / len(want) >= 0.7

    first = next((i for i, h in enumerate(hits) if any(on_page(h, e) for e in evidence)), None)
    top = hits[:k]
    return {"doc_hit": any(h["doc"] == e["doc"] for h in top for e in evidence),
            "page_hit": any(on_page(h, e) for h in top for e in evidence),
            "evidence_hit": any(has_text(h, e) for h in top for e in evidence),
            "mrr": 1.0 / (first + 1) if first is not None and first < k else 0.0}

## Run

In [ ]:
def doc_of(path):
    return path.rsplit("/", 1)[-1].removesuffix(".pdf")


with urllib.request.urlopen(QUESTIONS_URL) as f:
    questions = [json.loads(line) for line in f.read().decode().splitlines() if line.strip()]
for q in questions:  # evidence pages are 0-based in FinanceBench, 1-based in gold
    q["evidence"] = [{"doc": e["doc_name"], "page": int(e["evidence_page_num"]) + 1, "text": e["evidence_text"]}
                     for e in q["evidence"]]

paths = {doc_of(r.path): r.path for r in spark.table(n.children).select("path").distinct().collect()}
by_filing = defaultdict(list)
for q in questions:
    if q["doc_name"] in paths:
        by_filing[paths[q["doc_name"]]].append(q)
print(f"{len(questions)} questions, {sum(map(len, by_filing.values()))} with a chunked filing")

missing = spark.table(n.children).filter("embedding IS NULL").count()
assert missing == 0, f"{missing} chunks have no embedding: run the indexing job first"

run_at, summary = datetime.now(timezone.utc), []
try:
    if w("teardown_only") == "false":
        create_index()
        store = DatabricksVectorSearch(endpoint=ENDPOINT, index_name=INDEX, text_column="embed_text", columns=RETURNED,
                                       embedding=DatabricksEmbeddings(endpoint=w("embedding_endpoint")))
        for strategy in strategies:
            for mode in modes:
                results = {}
                for path, qs in by_filing.items():  # one retriever (and BM25 index) per filing
                    found = retriever_for(store, mode, path, strategy).batch(
                        [q["question"] for q in qs], config={"max_concurrency": 4})
                    for q, docs in zip(qs, found):
                        results[q["financebench_id"]] = [
                            {"doc": doc_of(d.metadata["path"]), "page_start": int(d.metadata["page_start"]),
                             "page_end": int(d.metadata["page_end"]), "text": d.page_content} for d in docs[:K]]
                for k in ks:
                    s = [score(results.get(q["financebench_id"], []), q["evidence"], k) for q in questions]
                    summary.append({"run_at": run_at, "strategy": strategy, "search": mode, "k": k,
                                    "questions": len(s), **{m: sum(x[m] for x in s) / len(s) for m in s[0]}})
                print(f"{strategy} / {mode}: done")
finally:
    if w("teardown") == "true" or w("teardown_only") == "true":
        teardown()
    else:
        print(f"teardown=false: endpoint {ENDPOINT} is still running and billed. Run with teardown_only=true.")

In [ ]:
if summary:
    df = spark.createDataFrame(summary).select("run_at", "strategy", "search", F.col("k").cast("int"),
                                               F.col("questions").cast("int"), "doc_hit", "page_hit",
                                               "evidence_hit", "mrr")
    df.write.mode("append").saveAsTable(n.retrieval_results)
    display(df.orderBy("k", F.desc("page_hit")))